In [1]:
import os
import pandas as pd
import re

# Initialize an empty dictionary to store the results
X_dict = {}

# Specify the directory to search for CSV files
directory = 'ibm_kyoto'

# Get a list of all CSV files in the specified directory
csv_files = [file for file in os.listdir(directory) if file.endswith('.csv') and file.startswith('ecr_error_')]

# Regular expression to extract (q_index1, q_index2) from the file name
pattern = re.compile(r'ecr_error_\d+_\((\d+),\s*(\d+)\)_y_pulse.csv')

for file in csv_files:
    match = pattern.match(file)
    if match:
        q_index1 = int(match.group(1))
        q_index2 = int(match.group(2))
        
        # Read the CSV file into a DataFrame
        df = pd.read_csv(os.path.join(directory, file))
        
        # Filter rows where the index is greater than 5
        filtered_df = df[df.index > 5]
        
        # Get the row with the minimum value in ['x0']
        min_x0_row = filtered_df.loc[(abs(filtered_df['x1']) + abs(filtered_df['x0'])).idxmin()]
        print((abs(filtered_df['x1']) + abs(filtered_df['x0'])).min())
        # Get the ['xval'] value from the row with the minimum ['x0']
        xval = min_x0_row['xval']
        
        # Store the result in the dictionary
        X_dict[(q_index1, q_index2)] = xval

0.008
0.0209999999999999
0.0104999999999999
0.015
0.006999999999999999
0.0169999999999998
0.0124999999999999


In [2]:
from functions.Train_ZNE import train_ZNE
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.circuit.library import RealAmplitudes,EfficientSU2

service = QiskitRuntimeService(channel='ibm_quantum',
    instance='ibm-q-skku/skku/skku-students',
    token = '06f802eeac992c43c4106753d7c5fc31414a13eb7d91c99d67fb49cc50569e5b4b011f20c71b9607a07cb43ecc7044557c699022db91d1685ba4c886d5886854')
backend = service.backend(directory)




In [3]:
from matplotlib import pyplot as plt
def plot_result(result,X_dict):
    for key in X_dict.keys():
        data_error = result[f'{list(key)}_even']
        data_ecr = result[f'{list(key)}_ecr']
        plt.figure()
        plt.title(key)
        plt.plot(data_error,'o-',label='error')
        plt.plot(data_ecr,'o-',label='ecr')
        plt.legend()
        plt.show()

In [5]:

stretch_list = [1,1.1,1.2,1.3,1.4]
class_list = []

for stretch in stretch_list:
    config = {'backend':backend,'x_amp_dict':X_dict,'error_stretch':stretch,'ecr_stretch':stretch}
    temp_class = train_ZNE(20,**config)
    temp_class.run()
    class_list.append(temp_class)
for index,class_name in enumerate(class_list):
    result = class_name.get_data()
    plot_result(result,X_dict)
    result.to_csv(f'./ecr_compare_result/{stretch_list[index]}_data_{directory}.csv')